# Sequence-to-Sequence (Seq2Seq): Attention Mechanisms & Translation Evaluation

This interactive lab covers:
1. **Encoder-Decoder Architecture**: The information bottleneck problem in fixed-length context vectors.
2. **Bahdanau vs. Luong Attention**: Additive vs. multiplicative dynamic alignment mechanisms.
3. **Teacher Forcing Dynamics**: Balancing training convergence against exposure bias.
4. **Evaluation Metrics**: Implementing sentence-level BLEU score with modified n-gram precision.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import torch
import torch.nn as nn
from seq2seq_engine import (
    BahdanauAttention, LuongAttention, Seq2SeqEncoder,
    Seq2SeqAttentionDecoder, Seq2SeqModel, compute_bleu_score
)

print('Seq2Seq & Attention engine loaded successfully!')

## 1. Comparing Attention Alignment (Bahdanau vs. Luong)

We inspect attention weights generated by both additive (Bahdanau) and multiplicative (Luong) scoring functions.

In [ ]:
torch.manual_seed(42)
B, src_len, enc_dim, dec_dim = 1, 5, 16, 16

dec_h = torch.randn(B, dec_dim)
enc_out = torch.randn(B, src_len, enc_dim)

bahdanau = BahdanauAttention(dec_dim, enc_dim)
luong = LuongAttention(dec_dim, enc_dim)

_, weights_b = bahdanau(dec_h, enc_out)
_, weights_l = luong(dec_h, enc_out)

print('Bahdanau Additive Attention Weights:', weights_b.squeeze().detach().numpy().round(3))
print('Luong Multiplicative Attention Weights:', weights_l.squeeze().detach().numpy().round(3))

## 2. Seq2Seq Forward Pass with Teacher Forcing

We pass source and target sequence batches through the end-to-end attention Seq2Seq model.

In [ ]:
src_vocab, tgt_vocab = 50, 60
embed_dim, hidden_dim = 16, 32

encoder = Seq2SeqEncoder(src_vocab, embed_dim, hidden_dim)
decoder = Seq2SeqAttentionDecoder(tgt_vocab, embed_dim, hidden_dim, hidden_dim)
model = Seq2SeqModel(encoder, decoder)

src_batch = torch.randint(0, src_vocab, (4, 10))  # 4 sequences, length 10
tgt_batch = torch.randint(0, tgt_vocab, (4, 8))   # 4 sequences, length 8

logits = model(src_batch, tgt_batch, teacher_forcing_ratio=0.5)
print(f'Input Source Batch: {src_batch.shape}')
print(f'Input Target Batch: {tgt_batch.shape}')
print(f'Output Logits:      {logits.shape} (Batch, TgtLen, TgtVocab)')
assert logits.shape == (4, 8, tgt_vocab)

## 3. BLEU Metric Evaluation for Translation

We evaluate machine translation candidate predictions against reference ground truth using our sentence BLEU implementation.

In [ ]:
reference = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
candidate1 = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
candidate2 = ['the', 'fast', 'brown', 'fox', 'leaps', 'over', 'a', 'lazy', 'hound']
candidate3 = ['totally', 'unrelated', 'output']

print(f'Candidate 1 (Exact):      BLEU-4 = {compute_bleu_score(reference, candidate1, max_n=4):.4f}')
print(f'Candidate 2 (Synonyms):   BLEU-2 = {compute_bleu_score(reference, candidate2, max_n=2):.4f}')
print(f'Candidate 3 (Unrelated):  BLEU-2 = {compute_bleu_score(reference, candidate3, max_n=2):.4f}')